# 📡 Aurora Forecasts — Data Retrieval

> **Author:** `<!-- your name -->`  · **Date:** `<!-- date -->`  · **Version:** 1.0

---

## Abstract

This notebook orchestrates the retrieval of Aurora Energy Research forecast scenario data via the Aurora HTTP API. It downloads **technology** and **system** scenario data at multiple time resolutions — **yearly (1y)**, **monthly (1m)**, and **hourly (1h, WIP)** — across configured European regions, persisting all results as Parquet files for downstream consumption.

Running this notebook is **Step 1** in the Aurora data pipeline. Downstream notebooks read the Parquet files produced here for aggregation, visualisation, and export to Excel.

---

## 🗺️ Notebook Map

| Step | Section | Description |
|---|---|---|
| 1 | Imports | Load `pandas` and the `AuroraAPI` retrieval client |
| 2 | Data Paths | Define Parquet and scenario-registry file paths per resolution |
| 3 | API Config | Read API token and country list from `api_params.yaml` |
| 4 | Yearly API | Initialise client, inspect available currencies, fetch yearly scenarios |
| 5 | Monthly API | Initialise client, fetch monthly scenarios |
| 6 | Hourly API | Initialise client — fetch step is currently a WIP |

> ⚙️ **Config source:** `config/aurora/api_params.yaml`
> 💾 **Output location:** `notebooks/data/aurora/` (Parquet files)

## 📦 1 · Imports & Environment Setup

Load standard data-manipulation libraries and the project-specific Aurora API client.

**Why here?** Centralising all imports at the top makes dependencies explicit and ensures the notebook fails fast if a package is missing.

| | |
|---|---|
| **Output** | `pd` (pandas) alias; `AuroraAPI` class ready for instantiation |

In [1]:
# Standard data-manipulation library — used for DataFrame inspection and type annotations
import pandas as pd

### Project Package Import

Import the `AuroraAPI` retrieval client from the `aurora_forecasts` package (installed in editable mode as part of this monorepo).

> 💡 **Tip:** If you see an `ImportError` here, run `pip install -e packages/aurora_forecasts` from the repository root.

In [2]:
# AuroraAPI: high-level client wrapping the Aurora Energy Research HTTP API.
# Handles authentication, incremental scenario registry management,
# chunked data retrieval, and persistence to Parquet.
from aurora_forecasts.retrieval_helper.retrieval_helper import AuroraAPI

## 🗂️ 2 · Configure Data Paths

Define file paths for scenario-component registries (JSON) and Parquet output databases for each time resolution.

**Why separate registries?** Each resolution (1y, 1m, 1h) maintains its own registry of already-downloaded scenario components to avoid redundant API calls on subsequent runs — only new or updated components are fetched.

| | |
|---|---|
| **Input** | Repository-relative path strings |
| **Output** | Path variables consumed by all `AuroraAPI` constructors below |

In [3]:
# --- Scenario-component registry paths (JSON) ---
# Each registry tracks which scenario components have already been fetched,
# preventing duplicate downloads on incremental re-runs.
aurora_registry_monthly_path = '../../../data/aurora/aurora_scenario_components_registry_1m.json'
aurora_registry_yearly_path  = '../../../data/aurora/aurora_scenario_components_registry_1y.json'
aurora_registry_hourly_path  = '../../../data/aurora/aurora_scenario_components_registry_1h.json'

# --- Parquet output paths — Yearly resolution ---
technology_path_yearly = '../../../data/aurora/aurora2_technology_scenarios_ES_default_currency_1y.parquet'
system_path_yearly     = '../../../data/aurora/aurora2_system_scenarios_ES_default_currency_1y.parquet'

# --- Parquet output paths — Monthly resolution ---
technology_path_monthly = '../../../data/aurora/aurora_technology_scenarios_ES_default_currency_1m.parquet'
system_path_monthly     = '../../../data/aurora/aurora_system_scenarios_ES_default_currency_1m.parquet'

# --- Parquet output paths — Hourly resolution ---
technology_path_hourly = '../../../data/aurora/aurora_technology_scenarios_ES_default_currency_1h.parquet'
system_path_hourly     = '../../../data/aurora/aurora_system_scenarios_ES_default_currency_1h.parquet'

# Path to the YAML config file containing the API token and country code list
api_config = '../../../config/aurora/api_params.yaml'

## 🔑 3 · Load API Configuration

Read the Aurora API token and target country codes from the YAML config file.

**Why YAML?** Externalising credentials to a config file keeps secrets out of version control and makes it easy to rotate the API token without touching notebook code.

| | |
|---|---|
| **Input** | `config/aurora/api_params.yaml` |
| **Output** | `aurora_token` (str), `country_code_list` (list[str]) |

In [4]:
import yaml

# Load the full YAML config block for the Aurora API
with open(api_config, 'r') as file:
    api_params = yaml.safe_load(file)

# Extract only the two fields needed for API authentication and geography targeting
aurora_token      = api_params['aurora_token']       # Bearer token for Aurora HTTP API
country_code_list = api_params['country_code_list']  # e.g. ['ES', 'DE', 'FR']

## 📅 4 · Yearly Resolution API — Initialise & Inspect

Instantiate an `AuroraAPI` client configured for yearly (1y) data, then inspect the available currency options.

**Why inspect currencies?** Aurora delivers prices in the currency specified at download time. Reviewing the currency catalogue confirms which base-year real values are available before committing to a full scenario pull.

| | |
|---|---|
| **Input** | `aurora_token`, `country_code_list`, yearly Parquet / registry paths |
| **Output** | `aurora_yr` client instance; `df_currencies` DataFrame displayed for reference |

In [5]:
# Instantiate the yearly AuroraAPI client.
# Passing country_code_list here sets the default geography for all operations
# on this instance; it can be overridden per-call in update_scenario_database().
aurora_yr = AuroraAPI(
    aurora_token,
    scenario_comp_registry_path=aurora_registry_yearly_path,
    technology_db_path=technology_path_yearly,
    country_code_list=country_code_list,
    system_db_path=system_path_yearly
    )

## ⬇️ 5 · Fetch Yearly Forecast Scenarios

Download and persist all yearly scenario components for the configured countries.

**How it works:** `update_scenario_database()` compares available scenario components from the API against the local registry. Only missing components are fetched and appended to the Parquet store — making incremental runs efficient.

| | |
|---|---|
| **Input** | `aurora_yr` client (already configured with paths and country list) |
| **Output** | Updated `technology_path_yearly` and `system_path_yearly` Parquet files |

> ⚠️ **Note:** This cell may take several minutes on first run when the local registry is empty.

In [6]:
# Trigger incremental yearly scenario download.
# Default resolution is '1y'; country list was set at construction time.
# Only scenario components absent from the local registry are fetched.
aurora_yr.update_scenario_database()

Retrieving data for scenario: Poland Nov 2020 (High)
No data retrieved for scenario: Poland Nov 2020 (High)
Retrieving data for scenario: Poland Nov 2020 (Low)
No data retrieved for scenario: Poland Nov 2020 (Low)
Scenario Iberia October 2020 (Central) for region prt already in registry, skipping retrieval.
Scenario Iberia October 2020 (High) for region prt already in registry, skipping retrieval.
Scenario Iberia October 2020 (Low) for region prt already in registry, skipping retrieval.
Scenario Iberia October 2020 (High) for region esp already in registry, skipping retrieval.
Scenario Iberia October 2020 (Low) for region esp already in registry, skipping retrieval.
Scenario Ireland Nov 2020 (Low) for region irx already in registry, skipping retrieval.
Scenario Ireland Nov 2020 (High) for region irx already in registry, skipping retrieval.
Scenario France Oct 2020 (Central) for region fra already in registry, skipping retrieval.
Scenario France Oct 2020 (Low) for region fra already in 

## 📆 6 · Monthly Resolution API — Initialise

Instantiate a second `AuroraAPI` client configured for monthly (1m) scenario retrieval.

| | |
|---|---|
| **Input** | `aurora_token`; monthly Parquet / registry paths |
| **Output** | `aurora` client instance ready for monthly fetch |

In [7]:
# Instantiate a second AuroraAPI client for monthly resolution.
# country_code_list is not set here — it will be passed explicitly to update_scenario_database().
aurora = AuroraAPI(
    aurora_token,
    scenario_comp_registry_path=aurora_registry_monthly_path,
    technology_db_path=technology_path_monthly,
    system_db_path=system_path_monthly
    )

## ⬇️ 7 · Fetch Monthly Forecast Scenarios

Download and persist all monthly scenario components for the configured countries.

| | |
|---|---|
| **Input** | `aurora` client; `country_code_list`; `resolution='1m'` |
| **Output** | Updated monthly Parquet files (see path note in §6) |

> ⚠️ **Note:** Monthly granularity produces significantly more data than yearly — first run may be slow.

In [ ]:
# Fetch monthly scenario data; resolution='1m' overrides the default '1y'.
# country_code_list is passed explicitly because it was not set at construction time.
aurora.update_scenario_database(
    resolution='1m',
    country_code_list=country_code_list
)

NameError: name 'aurora' is not defined

: 

: 

## ⏱️ 8 · Hourly Resolution API — Initialise & Single Forecast Retrieval

Instantiate the `AuroraAPI` client for hourly (1h) scenario retrieval, then fetch a single forecast for ad-hoc inspection.

**Status:** The bulk `update_scenario_database` call for hourly resolution is commented out — full batch download is a work in progress. `retrieve_single_forecast` is used instead to pull one specific scenario component for validation.

| | |
|---|---|
| **Input** | `aurora_token`, hourly Parquet / registry paths; scenario hash + parameters |
| **Output** | `aurora_h` client instance; `df` DataFrame with forecast values for the specified scenario |

In [ ]:
# Instantiate the hourly AuroraAPI client with hourly-specific Parquet and registry paths.
aurora_h = AuroraAPI(
    aurora_token,
    scenario_comp_registry_path=aurora_registry_hourly_path,
    technology_db_path=technology_path_hourly,
    system_db_path=system_path_hourly
)

# Bulk hourly fetch is currently disabled — update_scenario_database for resolution='1h'
# is a work in progress and may not yet handle hourly pagination correctly.
# aurora_h.update_scenario_database(
#     resolution='1h',
#     country_code_list=country_code_list
# )

# Fetch a single scenario component for ad-hoc inspection and validation.
# hash:          Aurora scenario identifier (obtain from available_scenarios_df)
# region_code:   lowercase Aurora region code ('esp' = Spain)
# sensitivity:   scenario sensitivity variant ('low', 'central', 'high')
# type:          data domain — 'system' (prices, emissions) or 'technology' (capacity, generation)
# currency_code: real-terms currency and base year (e.g. 'eur2024')
# resolution:    time granularity of the returned series ('1y', '1m', '1h')
# df = aurora_h.retrieve_single_forecast(
#     hash="92603186-3f51-4605-958d-4d49b6211b1c",
#     region_code="fra",
#     sensitivity="central",
#     type="system",
#     currency_code="eur2024",
#     resolution="1h"
# )

: 

In [ ]:
available_scenarios = aurora_h.available_scenarios_df
available_scenarios

,publishType,name,description,defaultCurrency,sensitivity,regionCode,hash,publicationDate,defaultFuturesDate,releaseDate,products,metaUrl,dataUrlBase
0,Published,Poland Nov 2020 (High),Poland November 2020 High,pln2019,high,pol,8e0a0094-7980-4343-8feb-218e66e6bd66,2021-01-14T12:28:31.000Z,None,2020-11-01T00:00:00.000Z,[pol_power_and_res],v1/scenarios/pmf/8e0a0094-7980-4343-8feb-218e6...,v1/scenarios/pmf/8e0a0094-7980-4343-8feb-218e6...
1,Published,Poland Nov 2020 (Low),Poland November 2020 Low,pln2019,low,pol,922dbf8b-2af1-414e-af68-18906a9c8203,2021-01-14T12:28:26.000Z,None,2020-11-01T00:00:00.000Z,[pol_power_and_res],v1/scenarios/pmf/922dbf8b-2af1-414e-af68-18906...,v1/scenarios/pmf/922dbf8b-2af1-414e-af68-18906...
2,Published,Iberia October 2020 (Central),Aurora's Central Scenario for October 2020 PMF...,eur2019,central,prt,ccd6c22b-cae9-4f6d-93d6-1213639802c9,2021-01-19T09:35:42.000Z,None,2020-10-01T00:00:00.000Z,"[ibr_power, ibr_power_and_res]",v1/scenarios/pmf/ccd6c22b-cae9-4f6d-93d6-12136...,v1/scenarios/pmf/ccd6c22b-cae9-4f6d-93d6-12136...
3,Published,Iberia October 2020 (High),Aurora's High scenario for October 2020 PMF in...,eur2019,high,prt,ba895fc3-9329-45c1-8d63-1af4ecdafb16,2021-01-19T09:34:41.000Z,None,2020-10-01T00:00:00.000Z,"[ibr_power, ibr_power_and_res]",v1/scenarios/pmf/ba895fc3-9329-45c1-8d63-1af4e...,v1/scenarios/pmf/ba895fc3-9329-45c1-8d63-1af4e...
4,Published,Iberia October 2020 (Low),Aurora's Low scenario for October 2020 PMF in ...,eur2019,low,prt,0a6d3698-1c18-48ea-840c-77ef7effe204,2021-01-19T09:33:45.000Z,None,2020-10-01T00:00:00.000Z,"[ibr_power, ibr_power_and_res]",v1/scenarios/pmf/0a6d3698-1c18-48ea-840c-77ef7...,v1/scenarios/pmf/0a6d3698-1c18-48ea-840c-77ef7...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1101,Published,Iberia Q1 26 (Central),This scenario does not include the Portuguese ...,eur2024,central,esp,b7058688-404d-4185-94fc-146825e036e2,2026-01-28T12:04:28.000Z,2026-01-28T00:00:00.000Z,2026-01-28T00:00:00.000Z,"[ibr_power_and_res, ibr_flex]",v1/scenarios/pmf/b7058688-404d-4185-94fc-14682...,v1/scenarios/pmf/b7058688-404d-4185-94fc-14682...
1102,Published,Iberia Q1 26 (Low),This scenario does not include the Portuguese ...,eur2024,low,prt,510abb49-4a79-4c0b-a342-21a5b349dde9,2026-01-28T12:05:08.000Z,2026-01-28T00:00:00.000Z,2026-01-28T00:00:00.000Z,"[ibr_power_and_res, ibr_flex]",v1/scenarios/pmf/510abb49-4a79-4c0b-a342-21a5b...,v1/scenarios/pmf/510abb49-4a79-4c0b-a342-21a5b...
1103,Published,Iberia Q1 26 (Low),This scenario does not include the Portuguese ...,eur2024,low,esp,510abb49-4a79-4c0b-a342-21a5b349dde9,2026-01-28T12:04:36.000Z,2026-01-28T00:00:00.000Z,2026-01-28T00:00:00.000Z,"[ibr_power_and_res, ibr_flex]",v1/scenarios/pmf/510abb49-4a79-4c0b-a342-21a5b...,v1/scenarios/pmf/510abb49-4a79-4c0b-a342-21a5b...
1104,Published,Iberia Q1 26 (High),This scenario does not include the Portuguese ...,eur2024,high,prt,1ec3619d-788e-450c-966c-2fcfdb83458f,2026-01-28T12:05:04.000Z,2026-01-28T00:00:00.000Z,2026-01-28T00:00:00.000Z,"[ibr_power_and_res, ibr_flex]",v1/scenarios/pmf/1ec3619d-788e-450c-966c-2fcfd...,v1/scenarios/pmf/1ec3619d-788e-450c-966c-2fcfd...


: 

In [ ]:
available_scenarios = aurora_h.available_scenarios_df
registry_list = []

for index, row in available_scenarios.iterrows():
    print(f"Release: {row['name']}")
    if (row['regionCode'] == 'esp') and ('25' in row['name'] and '26' in row['name']):
        df = aurora_h.retrieve_single_forecast(
            hash=row['hash'],
            region_code=row['regionCode'],
            sensitivity=row['sensitivity'],
            type='system',
            currency_code=row['defaultCurrency'],
            resolution="1h"
        )
        if not df.empty:
            registry_list.append({
                'region_code': row['regionCode'],
                'sensitivity': row['sensitivity'],
                'type': 'system',
                'currency_code': row['defaultCurrency']
            })

df_registry = pd.DataFrame(registry_list)
df_registry

Release: Poland Nov 2020 (High)
Release: Poland Nov 2020 (Low)
Release: Iberia October 2020 (Central)
Release: Iberia October 2020 (High)
Release: Iberia October 2020 (Low)
Release: Iberia October 2020 (High)
Release: Iberia October 2020 (Low)
Release: Ireland Nov 2020 (Low)
Release: Ireland Nov 2020 (High)
Release: France Oct 2020 (Central)
Release: France Oct 2020 (Low)
Release: Great Britain Oct 2020 (Central)
Release: Great Britain Oct 2020 (High)
Release: Great Britain  Oct 2020 (Low)
Release: France Oct 2020 (High)
Release: Great Britain Jan 2021 (High)
Release: Great Britain Jan 2021 (Low)
Release: Great Britain Jan 2021 (Central)
Release: Ireland Jan 2021 (High)
Release: Ireland Jan 2021 (Low)
Release: Ireland Jan 2021 (Central)
Release: Germany Oct 20 (Low)
Release: Poland Nov 2020 (Central)
Release: Iberia October 2020 (Central)
Release: Germany Oct 20 (Central)
Release: Germany Jan 21 (Central)
Release: Iberia January 2021 (Central)
Release: Germany Jan 21  (Low)
Release: Ib

C:\Users\mpy\Python Unified\packages\aurora_forecasts\src\aurora_forecasts\retrieval_helper\client.py:75: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(response.text))


                 Time (UTC) Wholesale market price
0                       UTC                EUR/MWh
1       2025-07-01 00:00:00                 125.24
2       2025-07-01 01:00:00                 125.87
3       2025-07-01 02:00:00                 126.19
4       2025-07-01 03:00:00                 126.62
...                     ...                    ...
311227  2060-12-31 18:00:00                  87.27
311228  2060-12-31 19:00:00                  86.82
311229  2060-12-31 20:00:00                  87.43
311230  2060-12-31 21:00:00                  85.69
311231  2060-12-31 22:00:00                  76.35

[311232 rows x 2 columns]
Release: Iberia Jan 25 (Origin Low)
Release: Iberia Jan 25 (Origin Low)


C:\Users\mpy\Python Unified\packages\aurora_forecasts\src\aurora_forecasts\retrieval_helper\client.py:75: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(response.text))


                 Time (UTC) Wholesale market price
0                       UTC                EUR/MWh
1       2025-04-01 00:00:00                  45.12
2       2025-04-01 01:00:00                  40.38
3       2025-04-01 02:00:00                  40.55
4       2025-04-01 03:00:00                  40.55
...                     ...                    ...
313411  2060-12-31 18:00:00                  59.09
313412  2060-12-31 19:00:00                  57.16
313413  2060-12-31 20:00:00                  56.38
313414  2060-12-31 21:00:00                  55.58
313415  2060-12-31 22:00:00                  39.75

[313416 rows x 2 columns]
Release: Iberia Jan 25 (Origin High)
Release: Iberia Jan 25 (Origin High)


...

C:\Users\mpy\Python Unified\packages\aurora_forecasts\src\aurora_forecasts\retrieval_helper\client.py:75: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(response.text))


                 Time (UTC)         Time (Local) Wholesale market price
0                       UTC            UTC+01:00                EUR/MWh
1       2025-12-31 23:00:00  2026-01-01 00:00:00                  43.99
2       2026-01-01 00:00:00  2026-01-01 01:00:00                  42.99
3       2026-01-01 01:00:00  2026-01-01 02:00:00                  42.33
4       2026-01-01 02:00:00  2026-01-01 03:00:00                  41.98
...                     ...                  ...                    ...
306812  2060-12-31 18:00:00  2060-12-31 19:00:00                   50.0
306813  2060-12-31 19:00:00  2060-12-31 20:00:00                  49.49
306814  2060-12-31 20:00:00  2060-12-31 21:00:00                  50.07
306815  2060-12-31 21:00:00  2060-12-31 22:00:00                  48.29
306816  2060-12-31 22:00:00  2060-12-31 23:00:00                  42.47

[306817 rows x 3 columns]
Release: Iberia Q1 26 (High)
Release: Iberia Q1 26 (High)
                 Time (UTC)         Time (Local) Wh

C:\Users\mpy\Python Unified\packages\aurora_forecasts\src\aurora_forecasts\retrieval_helper\client.py:75: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(response.text))


,region_code,sensitivity,type,currency_code
0,esp,central,system,eur2023
1,esp,low,system,eur2023
2,esp,high,system,eur2023
3,esp,centralextendedgenerationtax,system,eur2023
4,esp,lowextendedgenerationtax,system,eur2023
5,esp,central,system,eur2024
6,esp,low,system,eur2024
7,esp,high,system,eur2024
8,esp,lowextendedgenerationtax,system,eur2024
9,esp,centralextendedgenerationtax,system,eur2024


: 

In [ ]:
df_oct = aurora_h.retrieve_single_forecast(
    hash="5294aeed-978d-4e59-bfcb-2a359a9e2c50",
    region_code="esp",
    sensitivity="central",
    type="system",
    currency_code="eur2024",
    resolution="1h"
)

C:\Users\mpy\Python Unified\packages\aurora_forecasts\src\aurora_forecasts\retrieval_helper\client.py:75: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(response.text))


                 Time (UTC) Wholesale market price
0                       UTC                EUR/MWh
1       2025-12-31 23:00:00                  51.95
2       2026-01-01 00:00:00                  51.57
3       2026-01-01 01:00:00                  49.94
4       2026-01-01 02:00:00                  47.52
...                     ...                    ...
306812  2060-12-31 18:00:00                  82.35
306813  2060-12-31 19:00:00                  81.88
306814  2060-12-31 20:00:00                   82.5
306815  2060-12-31 21:00:00                  80.72
306816  2060-12-31 22:00:00                  79.58

[306817 rows x 2 columns]


C:\Users\mpy\Python Unified\packages\aurora_forecasts\src\aurora_forecasts\retrieval_helper\client.py:75: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(response.text))


                 Time (UTC) Wholesale market price
0                       UTC                EUR/MWh
1       2025-06-30 23:00:00                 111.24
2       2025-07-01 00:00:00                 111.43
3       2025-07-01 01:00:00                 112.08
4       2025-07-01 02:00:00                 112.41
...                     ...                    ...
311228  2060-12-31 18:00:00                  93.09
311229  2060-12-31 19:00:00                  92.63
311230  2060-12-31 20:00:00                  93.25
311231  2060-12-31 21:00:00                  85.15
311232  2060-12-31 22:00:00                  81.24

[311233 rows x 2 columns]


: 

In [ ]:
# print(df_jul[df_jul['datetime'] >= '2026-01-01'].head())
print(df_oct[df_oct['datetime'] >= '2026-01-01'].head())
# print(df_ene[df_ene['datetime'] >= '2026-01-01'].head())
df_jul

             datetime                variable  value    units
1 2026-01-01 00:00:00  Wholesale market price  51.57  EUR/MWh
2 2026-01-01 01:00:00  Wholesale market price  49.94  EUR/MWh
3 2026-01-01 02:00:00  Wholesale market price  47.52  EUR/MWh
4 2026-01-01 03:00:00  Wholesale market price  46.65  EUR/MWh
5 2026-01-01 04:00:00  Wholesale market price  49.49  EUR/MWh


,datetime,variable,value,units
0,2025-06-30 23:00:00,Wholesale market price,111.24,EUR/MWh
1,2025-07-01 00:00:00,Wholesale market price,111.43,EUR/MWh
2,2025-07-01 01:00:00,Wholesale market price,112.08,EUR/MWh
3,2025-07-01 02:00:00,Wholesale market price,112.41,EUR/MWh
4,2025-07-01 03:00:00,Wholesale market price,112.84,EUR/MWh
...,...,...,...,...
311227,2060-12-31 18:00:00,Wholesale market price,93.09,EUR/MWh
311228,2060-12-31 19:00:00,Wholesale market price,92.63,EUR/MWh
311229,2060-12-31 20:00:00,Wholesale market price,93.25,EUR/MWh
311230,2060-12-31 21:00:00,Wholesale market price,85.15,EUR/MWh


: 

In [ ]:
# comprobar si la actualizacion es trimestral o si varía cada trimestre
# en caso de que sea trimestral, se están colando
# si es semestral encaja porque sería el flex, lo que tenemos contratado


: 

In [ ]:
df_r = aurora_h.available_scenarios_df
df_r

,publishType,name,description,defaultCurrency,sensitivity,regionCode,hash,publicationDate,defaultFuturesDate,releaseDate,products,metaUrl,dataUrlBase
0,Published,Poland Nov 2020 (High),Poland November 2020 High,pln2019,high,pol,8e0a0094-7980-4343-8feb-218e66e6bd66,2021-01-14T12:28:31.000Z,None,2020-11-01T00:00:00.000Z,[pol_power_and_res],v1/scenarios/pmf/8e0a0094-7980-4343-8feb-218e6...,v1/scenarios/pmf/8e0a0094-7980-4343-8feb-218e6...
1,Published,Poland Nov 2020 (Low),Poland November 2020 Low,pln2019,low,pol,922dbf8b-2af1-414e-af68-18906a9c8203,2021-01-14T12:28:26.000Z,None,2020-11-01T00:00:00.000Z,[pol_power_and_res],v1/scenarios/pmf/922dbf8b-2af1-414e-af68-18906...,v1/scenarios/pmf/922dbf8b-2af1-414e-af68-18906...
2,Published,Iberia October 2020 (Central),Aurora's Central Scenario for October 2020 PMF...,eur2019,central,prt,ccd6c22b-cae9-4f6d-93d6-1213639802c9,2021-01-19T09:35:42.000Z,None,2020-10-01T00:00:00.000Z,"[ibr_power, ibr_power_and_res]",v1/scenarios/pmf/ccd6c22b-cae9-4f6d-93d6-12136...,v1/scenarios/pmf/ccd6c22b-cae9-4f6d-93d6-12136...
3,Published,Iberia October 2020 (High),Aurora's High scenario for October 2020 PMF in...,eur2019,high,prt,ba895fc3-9329-45c1-8d63-1af4ecdafb16,2021-01-19T09:34:41.000Z,None,2020-10-01T00:00:00.000Z,"[ibr_power, ibr_power_and_res]",v1/scenarios/pmf/ba895fc3-9329-45c1-8d63-1af4e...,v1/scenarios/pmf/ba895fc3-9329-45c1-8d63-1af4e...
4,Published,Iberia October 2020 (Low),Aurora's Low scenario for October 2020 PMF in ...,eur2019,low,prt,0a6d3698-1c18-48ea-840c-77ef7effe204,2021-01-19T09:33:45.000Z,None,2020-10-01T00:00:00.000Z,"[ibr_power, ibr_power_and_res]",v1/scenarios/pmf/0a6d3698-1c18-48ea-840c-77ef7...,v1/scenarios/pmf/0a6d3698-1c18-48ea-840c-77ef7...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1101,Published,Iberia Q1 26 (Central),This scenario does not include the Portuguese ...,eur2024,central,esp,b7058688-404d-4185-94fc-146825e036e2,2026-01-28T12:04:28.000Z,2026-01-28T00:00:00.000Z,2026-01-28T00:00:00.000Z,"[ibr_power_and_res, ibr_flex]",v1/scenarios/pmf/b7058688-404d-4185-94fc-14682...,v1/scenarios/pmf/b7058688-404d-4185-94fc-14682...
1102,Published,Iberia Q1 26 (Low),This scenario does not include the Portuguese ...,eur2024,low,prt,510abb49-4a79-4c0b-a342-21a5b349dde9,2026-01-28T12:05:08.000Z,2026-01-28T00:00:00.000Z,2026-01-28T00:00:00.000Z,"[ibr_power_and_res, ibr_flex]",v1/scenarios/pmf/510abb49-4a79-4c0b-a342-21a5b...,v1/scenarios/pmf/510abb49-4a79-4c0b-a342-21a5b...
1103,Published,Iberia Q1 26 (Low),This scenario does not include the Portuguese ...,eur2024,low,esp,510abb49-4a79-4c0b-a342-21a5b349dde9,2026-01-28T12:04:36.000Z,2026-01-28T00:00:00.000Z,2026-01-28T00:00:00.000Z,"[ibr_power_and_res, ibr_flex]",v1/scenarios/pmf/510abb49-4a79-4c0b-a342-21a5b...,v1/scenarios/pmf/510abb49-4a79-4c0b-a342-21a5b...
1104,Published,Iberia Q1 26 (High),This scenario does not include the Portuguese ...,eur2024,high,prt,1ec3619d-788e-450c-966c-2fcfdb83458f,2026-01-28T12:05:04.000Z,2026-01-28T00:00:00.000Z,2026-01-28T00:00:00.000Z,"[ibr_power_and_res, ibr_flex]",v1/scenarios/pmf/1ec3619d-788e-450c-966c-2fcfd...,v1/scenarios/pmf/1ec3619d-788e-450c-966c-2fcfd...


: 

---

## 🏁 Summary & Next Steps

### What We Did
- Imported `pandas` and the `AuroraAPI` retrieval client from the `aurora_forecasts` package.
- Defined all Parquet output paths and scenario-registry paths for yearly, monthly, and hourly resolutions.
- Loaded the Aurora API token and target country list from `config/aurora/api_params.yaml`.
- Initialised a yearly `AuroraAPI` client, inspected the available currency catalogue, and fetched yearly forecast scenarios.
- Initialised a monthly `AuroraAPI` client and fetched monthly forecast scenarios.
- Initialised an hourly `AuroraAPI` client and retrieved a single forecast for ad-hoc validation (bulk hourly fetch pending).

### Output Files Produced

| Resolution | Technology Parquet | System Parquet |
|---|---|---|
| Yearly (1y) | `aurora2_technology_scenarios_ES_default_currency_1y.parquet` | `aurora2_system_scenarios_ES_default_currency_1y.parquet` |
| Monthly (1m) | `aurora_technology_scenarios_ES_default_currency_1m.parquet` | `aurora_system_scenarios_ES_default_currency_1m.parquet` |
| Hourly (1h) | `aurora_technology_scenarios_ES_default_currency_1h.parquet` *(WIP)* | `aurora_system_scenarios_ES_default_currency_1h.parquet` *(WIP)* |

### Suggested Next Steps
1. **Complete hourly fetch** — implement and test `update_scenario_database(resolution='1h', ...)` in `AuroraAPI`.
2. **Proceed to downstream notebook** — load the Parquet files produced here for data cleaning, aggregation, and visualisation.

---
*Generated documentation · feel free to edit and expand.*